In [ ]:
from pathlib import Path

import logfire

from src.common.utils.constants import ParseMethod
from src.common.utils.helper import separate_content
from src.common.utils.tokenizer import TikTokenTokenizer
from src.ingestion.chunking.chunk import build_parent_child_chunk
from src.ingestion.chunking.chunker_factory import create_chunker
from src.ingestion.chunking.chunking_config import ChunkingConfig
from src.ingestion.processor import Processor

In [ ]:
logfire.configure(service_name="parsing")

In [ ]:
cwd = Path.cwd().parent
file_path = cwd / "data/Attention-is_all_you_need.pdf"

In [ ]:
tokenizer = TikTokenTokenizer()
processor = Processor(tokenizer)

In [ ]:
out, doc_id = await processor.process_document(
    file_path=str(file_path), parse_method=ParseMethod.DOCLING
)

In [ ]:
print(out)

In [ ]:
text_content, multimodal_items = separate_content(out)

In [ ]:
multimodal_items

In [ ]:
chunking_config = ChunkingConfig(type="recursive_character", size=512, overlap=64)
chunker = create_chunker(chunking_config)
text_chunks = chunker.chunk(text_content)
multimodel_chunks = chunker.chunk_multimodal_items(
    multimodal_items, doc_id=doc_id, source_file=str(file_path), start_index=len(text_chunks)
)

In [ ]:
chunks = text_chunks + multimodel_chunks

In [ ]:
len(chunks)

In [ ]:
enriched_chunks = build_parent_child_chunk(chunks, tokenizer)

In [ ]:
enriched_chunks

In [ ]:
result = await processor.ingest_document(file_path=file_path, parse_method=ParseMethod.DOCLING)

In [ ]:
result